# 🧪 W6-D5 概念实验：从"玩具 Agent"到"生产 Agent"，差距到底在哪？

> 配套阅读：`ima/第6周-Day5-Agent开发实战与框架设计.md`（框架横评、选型决策树、状态管理对比表在那边）
> 本 notebook 不搬运 md，只用可执行实验回答三个问题：
>
> 1. **指数退避重试**为什么优于固定间隔？为什么参数错误绝对不能重试？
> 2. 对话历史无限增长时，"漏斗式"记忆管理到底省多少 token？
> 3. 可观测性里的 trace / span 记录的是什么东西？
>
> 实验环境：纯 Python 标准库 + numpy 模拟，不调用任何 LLM API。

## 实验 1：重试策略的分类学——"暂时性错误"重试，"参数错误"不重试

生产环境的工具调用一定会失败。md 里给了四种错误类型，这里用代码验证最关键的分水岭：

- **暂时性错误**（超时/过载）：过一会儿再试，可能就好了 → 重试
- **参数错误**：同样的错误参数再发一万次，还是同样的错 → 重试无意义，应反馈修正或降级

用一个"前两次超时、第三次成功"的 mock 工具 + 一个"参数天生非法"的调用来对比。

In [ ]:
import time
from datetime import datetime

class RobustToolExecutor:
    """带错误分类的重试执行器（模拟时钟，不真 sleep）"""

    def __init__(self, max_retries=4):
        self.max_retries = max_retries
        self.clock = 0.0                      # 模拟时钟
        self.call_log = []                    # 可观测性：每次尝试都留痕

    def execute(self, tool_name, args, tool_fn, fallback_fn=None):
        for attempt in range(1, self.max_retries + 1):
            try:
                t0 = time.perf_counter()
                result = tool_fn(**args)
                self.call_log.append((tool_name, attempt, "success", time.perf_counter() - t0))
                return {"status": "success", "data": result, "attempts": attempt, "t": self.clock}
            except ValueError as e:
                # 参数错误：确定性错误，重试没有意义 → 立即返回让上层修正
                self.call_log.append((tool_name, attempt, "param_error(no_retry)", 0.0))
                return {"status": "param_error", "error": str(e)}
            except TimeoutError:
                # 暂时性错误：指数退避后重试 1s → 2s → 4s → 8s
                self.call_log.append((tool_name, attempt, "timeout", 0.0))
                if attempt < self.max_retries:
                    backoff = 2 ** (attempt - 1)
                    self.clock += backoff
        # 全部重试失败 → 降级方案（不是崩溃，而是兜底）
        if fallback_fn is not None:
            self.call_log.append((tool_name, "-", "fallback", 0.0))
            return {"status": "fallback", "data": fallback_fn(**args)}
        return {"status": "failed"}

# --- mock 工具 1：前 2 次超时，第 3 次成功（模拟服务过载恢复）---
state = {"calls": 0}
def flaky_weather_api(city):
    state["calls"] += 1
    if state["calls"] <= 2:
        raise TimeoutError("service overloaded")
    return f"{city}: 32°C 晴"

# --- mock 工具 2：LLM 提取的参数天生非法 ---
def get_weather(city: str):
    if not isinstance(city, str) or len(city) == 0:
        raise ValueError("invalid param: city")
    return f"{city}: 30°C 多云"

ex = RobustToolExecutor()
print("场景 A：暂时性超时 → 指数退避重试")
r1 = ex.execute("get_weather", {"city": "北京"}, flaky_weather_api)
print(f"  结果: {r1['status']}，尝试 {r1['attempts']} 次，模拟耗时 {r1['t']}s → {r1['data']}")

print("\n场景 B：参数错误 → 不重试，直接返回错误")
r2 = ex.execute("get_weather", {"city": ""}, get_weather)
print(f"  结果: {r2['status']}，错误: {r2['error']}")
print(f"  重试次数: 0（同样的输入必然产生同样的错误）")

print("\n场景 C：永久故障 → 重试耗尽后降级兜底")
def always_down(city): raise TimeoutError("down")
def cached_fallback(city): return f"{city}: [缓存数据] 31°C（可能有延迟）"
r3 = ex.execute("get_weather", {"city": "上海"}, always_down, cached_fallback)
print(f"  结果: {r3['status']} → {r3['data']}")

print("\n📋 调用日志（可观测性最原始的形态）:")
for entry in ex.call_log:
    print(f"  {datetime.now().strftime('%H:%M:%S')} [{entry[2]:>20s}] {entry[0]} attempt={entry[1]}")

## 实验 2：指数退避 vs 固定间隔——对故障中的服务温柔一点

直觉问题：既然固定间隔 1s 重试最终也能成功，为什么要 1s→2s→4s？

关键在**故障期间发出的无效请求量**。服务过载时，每次重试都是往一辆熄火的车上再踩一脚油门。
用确定性模拟对比两种策略：服务在时刻 T 恢复（横轴），统计恢复前客户端白发了多少请求、恢复后又等了多久。

In [ ]:
import numpy as np

FIXED  = np.arange(1.0, 16.0, 1.0)     # 固定间隔：每 1 秒重试一次（1,2,...,15 秒）
EXP    = np.cumsum([1, 2, 4, 8, 16])   # 指数退避：第1/3/7/15/31秒各重试一次

T = np.linspace(0.01, 14, 400)         # 假设：服务恢复时刻 T（0~14 秒均匀取值）

def wasted(retries, T):
    """恢复前发出的无效重试次数"""
    return (retries[None, :] < T[:, None]).sum(axis=1)

def latency(retries, T):
    """恢复后到下一次成功重试的等待时间"""
    ok = retries[None, :] >= T[:, None]
    first = np.where(ok.any(axis=1), ok.argmax(axis=1), len(retries) - 1)
    return retries[first] - T

w_fixed, w_exp = wasted(FIXED, T), wasted(EXP, T)
l_fixed, l_exp = latency(FIXED, T), latency(EXP, T)

print(f"服务恢复时刻 T=10s 时：")
i = np.searchsorted(T, 10.0)
print(f"  固定间隔：恢复前白发 {w_fixed[i]} 个请求，恢复后又等 {l_fixed[i]:.1f}s")
print(f"  指数退避：恢复前白发 {w_exp[i]} 个请求，恢复后又等 {l_exp[i]:.1f}s")
print()
print(f"全部恢复时刻平均：固定间隔白发 {w_fixed.mean():.1f} 个请求 vs 指数退避 {w_exp.mean():.1f} 个")
print(f"                 固定间隔平均多等 {l_fixed.mean():.2f}s vs 指数退避 {l_exp.mean():.2f}s")
print()
print("结论：指数退避用『恢复后稍等久一点』换『故障期间请求量对数级下降』。")
print("     如果失败原因是过载，减少无效请求本身就是帮服务更快恢复。")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))

ax1.plot(T, w_fixed, label="固定间隔 1s", color="#EF5350")
ax1.plot(T, w_exp, label="指数退避 1→2→4→8s", color="#42A5F5")
ax1.set_xlabel("服务恢复时刻 T（秒）")
ax1.set_ylabel("恢复前的无效重试次数")
ax1.set_title("故障期间的白发请求量（越少越好）")
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(T, l_fixed, label="固定间隔 1s", color="#EF5350")
ax2.plot(T, l_exp, label="指数退避 1→2→4→8s", color="#42A5F5")
ax2.set_xlabel("服务恢复时刻 T（秒）")
ax2.set_ylabel("恢复后额外等待（秒）")
ax2.set_title("代价：恢复后可能多等一会儿")
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()
print("读图：左图指数退避呈阶梯状（请求量对数增长），右图是它的代价——锯齿形等待。")
print("这就是『给故障服务喘息时间』的定量版。")

## 实验 3：漏斗式记忆管理——上下文无限增长 vs 滚动摘要

md 里的"漏斗模式"：全量对话历史（短期）→ 压缩摘要（中期）→ 知识沉淀（长期）。

模拟一个 20 轮的对话，每轮产生 user(30 tok) + 工具结果(200 tok) + 回复(50 tok)。
对比两种策略下，**每一轮发给 LLM 的上下文 token 数**：

- 不管理：历史全量塞进 prompt，线性膨胀
- 漏斗管理：只保留最近 3 轮原文，更早的每轮压缩成 20 token 摘要

In [ ]:
import numpy as np

N_ROUNDS, KEEP_FULL = 20, 3
TURN_TOKENS = {"user": 30, "tool_result": 200, "reply": 50}   # 每轮 280 token
SUMMARY_TOKENS = 20                                           # 压缩后每轮只留 20

per_round = sum(TURN_TOKENS.values())

tokens_no_mgmt = [r * per_round for r in range(1, N_ROUNDS + 1)]

def funnel_tokens(round_idx):
    older = max(round_idx - KEEP_FULL, 0)      # 被摘要的轮数
    full = min(round_idx, KEEP_FULL)           # 保留原文的轮数
    return older * SUMMARY_TOKENS + full * per_round

tokens_funnel = [funnel_tokens(r) for r in range(1, N_ROUNDS + 1)]

for r in [5, 10, 20]:
    print(f"第 {r:>2} 轮：不管理 {tokens_no_mgmt[r-1]:>5} tok | 漏斗 {tokens_funnel[r-1]:>4} tok "
          f"| 省 {1 - tokens_funnel[r-1]/tokens_no_mgmt[r-1]:.0%}")
total_no = sum(tokens_no_mgmt)
total_f = sum(tokens_funnel)
print(f"\n20 轮累计发送量：不管理 {total_no:,} tok vs 漏斗 {total_f:,} tok（省 {1-total_f/total_no:.0%}）")
print("\n注意：省的不只是钱——上下文越长，注意力越分散、响应越慢（lost in the middle）。")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

rounds = np.arange(1, N_ROUNDS + 1)
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(rounds, tokens_no_mgmt, "o-", color="#EF5350", label="不管理（全量历史）")
ax.plot(rounds, tokens_funnel, "s-", color="#42A5F5", label=f"漏斗（保留最近{KEEP_FULL}轮 + 摘要）")
ax.set_xlabel("对话轮数")
ax.set_ylabel("每轮发送给 LLM 的上下文 token")
ax.set_title("漏斗式记忆管理：线性膨胀 vs 恒定-ish 上下文")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print("读图：不管理是直线上升；漏斗在超过 3 轮后变成缓坡（斜率 = 20 tok/轮 而不是 280 tok/轮）。")

## 实验 4：可观测性——trace/span 到底长什么样

md 里说可观测性三支柱是"日志 + 追踪 + 指标"。追踪的核心数据结构就是 **trace（一次请求）→ span（请求内的每一步）**。

用一个 mini Agent 跑两次请求，给每一步打 span，然后输出追踪报告——这就是 LangSmith / OpenTelemetry 的骨架。

In [ ]:
import time
from datetime import datetime

class Tracer:
    """最小追踪器：trace = 一次用户请求，span = 其中一步"""
    def __init__(self):
        self.traces = []

    @property
    def current(self): return self.traces[-1]

    def start(self, user_message):
        self.traces.append({"msg": user_message, "spans": [], "t0": time.perf_counter()})

    def span(self, name, kind, **data):
        t = time.perf_counter()
        self.current["spans"].append({"name": name, "kind": kind, "data": data, "t": t})

    def end(self): self.current["total_s"] = time.perf_counter() - self.current["t0"]

def busy(ms):  # 用纯计算模拟每步耗时
    x = 0
    for _ in range(int(ms * 20000)): x += 1
    return x

tracer = Tracer()
MOCK_DB = {"北京": "32°C 晴", "上海": "28°C 多云"}

def agent_chat(msg):
    """一个最小 Agent 请求：决策 → 工具 → 整合回复"""
    tracer.start(msg)
    tracer.span("LLM 决策", "decision", tool="get_weather")
    busy(8)
    city = next((c for c in MOCK_DB if c in msg), "北京")   # mock 参数提取
    tracer.span("get_weather", "tool_call", city=city)
    busy(3)
    data = MOCK_DB.get(city, "暂无数据")
    tracer.span("整合回复", "response", chars=len(data))
    busy(4)
    tracer.end()
    return f"北京今天 {data}"

agent_chat("北京天气怎么样？")
agent_chat("上海呢？")

icons = {"decision": "🧠", "tool_call": "🔧", "response": "💬"}
for tr in tracer.traces:
    print(f"\n📋 Trace: 「{tr['msg']}」  总耗时 {tr['total_s']*1000:.1f}ms")
    for sp in tr["spans"]:
        dt = (sp["t"] - tr["t0"]) * 1000
        print(f"   {icons[sp['kind']]} {sp['name']:<12s} +{dt:6.1f}ms  {sp['data']}")

print("\n这个输出就是『指标』的数据源：span 耗时可以聚合出 P95 延迟、工具成功率等。")
print("没有 trace 的 Agent 在生产环境就是黑盒——出了问题只能靠猜。")

## 结论

| 实验 | 验证的概念 | 一句话 |
|---|---|---|
| 1 | 错误分类处理 | 暂时性错误重试；参数错误确定性失败，重试无意义 |
| 2 | 指数退避 | 用恢复后的等待换故障期请求量对数下降 |
| 3 | 漏斗记忆管理 | 上下文从 280 tok/轮 的膨胀降到 20 tok/轮 |
| 4 | 可观测性 | trace→span 是生产 Agent 的黑匣子记录仪 |

**框架不是"更强的 Agent"，是 Agent 的运行环境**：流程控制、状态管理、错误处理、可观测性。

→ 深入阅读：同名 md 第 2-3 节（状态管理策略表、五大框架横评、选型决策树）